# Pandas Session 3 Assignment: Superstore Sales Analysis

## GroupBy, Aggregation, Time Series and Pivot Tables

**Dataset:** Sample Superstore Sales

**Source:** Kaggle - https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

---

### Learning Objectives

This assignment tests your understanding of Session 3 concepts:

- GroupBy operations (split-apply-combine)
- Aggregation with agg() and named aggregations
- Transform vs aggregation
- Pivot tables and crosstab
- Time series operations (to_datetime, resample, rolling)
- Shift for lag features

---

### Dataset Description

The Superstore dataset contains sales transactions from a retail store.

**Key Columns:**

- Order ID, Order Date, Ship Date - Transaction info
- Customer ID, Customer Name, Segment - Customer info
- Country, City, State, Region - Geography
- Product ID, Category, Sub-Category, Product Name - Product info
- Sales, Quantity, Discount, Profit - Metrics

---

### Business Context

You are a data analyst at Superstore. Management wants answers to:

- Which regions/categories are most profitable?
- What are the sales trends over time?
- Which customer segments perform best?
- How can we identify top and bottom performers?

**Total Points: 100 (+ 10 bonus)**

---
## Part 1: Data Loading and Preparation (10 points)
---

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("Sample - Superstore.csv", encoding="latin-1")

print("Dataset loaded!")
print(f"Shape: {df.shape}")

df.head()

Dataset loaded!
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Task 1.1: Data Preparation (5 points)

1. Convert 'Order Date' and 'Ship Date' to datetime
2. Create 'Order_Year', 'Order_Month', 'Order_Quarter' columns from Order Date
3. Set 'Order Date' as the index (keep a copy of the original df)

In [10]:
df["Order Date"] = pd.to_datetime(df["Order Date"])

df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Order_Year"] = df["Order Date"].dt.year

df["Order_Month"] = df["Order Date"].dt.month

df["Order_Quarter"] = df["Order Date"].dt.quarter

df_original = df.copy()

df = df.set_index("Order Date")

df.head()

,Row ID,Order ID,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order_Year,Order_Month,Order_Quarter
Order Date,,,,,,,,,,,,,,,,,,,,,
2016-11-08,1,CA-2016-152156,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,4
2016-11-08,2,CA-2016-152156,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,4
2016-06-12,3,CA-2016-138688,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,2
2015-10-11,4,US-2015-108966,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,4
2015-10-11,5,US-2015-108966,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,4


### Task 1.2: Initial Exploration (5 points)

1. How many unique customers, products, and orders?
2. What is the date range of the data?
3. What are the unique values in Category and Region?

In [11]:
df["Customer ID"].nunique()

df["Product ID"].nunique()

df["Order ID"].nunique()

df.index.min(), df.index.max()

df["Category"].unique()

df["Region"].unique()

<ArrowStringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str

---
## Part 2: Basic GroupBy Operations (20 points)
---

### Task 2.1: Single Column GroupBy (5 points)

1. Calculate total Sales by Region
2. Calculate average Profit by Category
3. Count number of orders by Segment

In [12]:
df.groupby("Region")["Sales"].sum()

df.groupby("Category")["Profit"].mean()

df.groupby("Segment")["Order ID"].count()

Segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: Order ID, dtype: int64

### Task 2.2: Multiple Column GroupBy (5 points)

1. Calculate total Sales by Region AND Category
2. Calculate average Discount by Category AND Sub-Category
3. Find the count of orders by Year AND Quarter

In [13]:
df.groupby(["Region", "Category"])["Sales"].sum()

df.groupby(["Category", "Sub-Category"])["Discount"].mean()

df.groupby(["Order_Year", "Order_Quarter"])["Order ID"].count()

Order_Year  Order_Quarter
2014        1                 282
            2                 392
            3                 564
            4                 755
2015        1                 260
            2                 444
            3                 592
            4                 806
2016        1                 335
            2                 594
            3                 740
            4                 918
2017        1                 500
            2                 690
            3                 903
            4                1219
Name: Order ID, dtype: int64

### Task 2.3: GroupBy with Multiple Aggregations (5 points)

For each Category, calculate:

- Total Sales (sum)
- Average Profit (mean)
- Number of transactions (count)
- Maximum single sale (max)

In [14]:
df.groupby("Category").agg({
    "Sales": "sum",
    "Profit": "mean",
    "Order ID": "count",
    "Sales": "max"
})

,Sales,Profit,Order ID
Category,,,
Furniture,4416.174,8.699327,2121
Office Supplies,9892.740,20.327050,6026
Technology,22638.480,78.752002,1847


### Task 2.4: Named Aggregations (5 points)

Use named aggregations to create a clean summary by Region:

- total_sales: sum of Sales
- avg_profit: mean of Profit
- total_quantity: sum of Quantity
- order_count: count of Order ID
- avg_discount: mean of Discount

In [15]:
df.groupby("Region").agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    total_quantity=("Quantity", "sum"),
    order_count=("Order ID", "count"),
    avg_discount=("Discount", "mean")
)

,total_sales,avg_profit,total_quantity,order_count,avg_discount
Region,,,,,
Central,501239.8908,17.092709,8780,2323,0.240353
East,678781.2400,32.135808,10618,2848,0.145365
South,391721.9050,28.857673,6209,1620,0.147253
West,725457.8245,33.849032,12266,3203,0.109335


---
## Part 3: Advanced Aggregation (15 points)
---

### Task 3.1: Custom Aggregation Functions (5 points)

1. Calculate the profit margin (Profit/Sales * 100) for each Category
2. Find the range (max - min) of Sales for each Region
3. Calculate the coefficient of variation (std/mean) of Profit by Segment

In [16]:
profit_margin = df.groupby("Category").apply(
    lambda x: (x["Profit"].sum() / x["Sales"].sum()) * 100
)

sales_range = df.groupby("Region")["Sales"].agg(
    lambda x: x.max() - x.min()
)

profit_cv = df.groupby("Segment")["Profit"].agg(
    lambda x: x.std() / x.mean()
)

profit_margin

sales_range

profit_cv

Segment
Consumer       9.389450
Corporate      7.616929
Home Office    6.280008
Name: Profit, dtype: float64

### Task 3.2: Top N Analysis (5 points)

1. Find the top 5 customers by total Sales
2. Find the top 3 products by total Profit in each Category
3. Find the bottom 5 Sub-Categories by average Profit

In [17]:
top_5_customers = df.groupby("Customer Name")["Sales"].sum().nlargest(5)

top_3_products = (
    df.groupby(["Category", "Product Name"])["Profit"]
    .sum()
    .groupby(level=0)
    .nlargest(3)
)

bottom_5_subcategories = (
    df.groupby("Sub-Category")["Profit"]
    .mean()
    .nsmallest(5)
)

top_5_customers

top_3_products

bottom_5_subcategories

Sub-Category
Tables      -55.565771
Bookcases   -15.230509
Supplies     -6.258418
Fasteners     4.375660
Art           8.200737
Name: Profit, dtype: float64

### Task 3.3: Filter Groups (5 points)

1. Filter to only Categories with total Sales > 500,000
2. Filter to only States with more than 100 orders
3. Find Customers who have made purchases in all 4 Regions

In [18]:
high_sales_categories = (
    df.groupby("Category")["Sales"]
    .sum()
    .loc[lambda x: x > 500000]
)

states_more_100_orders = (
    df.groupby("State")["Order ID"]
    .count()
    .loc[lambda x: x > 100]
)

customers_all_regions = (
    df.groupby("Customer Name")["Region"]
    .nunique()
    .loc[lambda x: x == 4]
)

high_sales_categories

states_more_100_orders

customers_all_regions

Customer Name
Aaron Smayling        4
Adam Bellavance       4
Adam Hart             4
Adam Shillingsburg    4
Adrian Barton         4
                     ..
Victoria Pisteka      4
Victoria Wilson       4
Vivian Mathis         4
Xylona Preis          4
Zuschuss Carroll      4
Name: Region, Length: 301, dtype: int64

---
## Part 4: Transform Operations (15 points)
---

### Task 4.1: Basic Transform (5 points)

1. Add a column 'Category_Avg_Sales' showing the average sales for that row's category
2. Add a column 'Region_Total_Profit' showing total profit for that row's region
3. Verify that transform returns same-length output as input

In [19]:
df["Category_Avg_Sales"] = df.groupby("Category")["Sales"].transform("mean")

df["Region_Total_Profit"] = df.groupby("Region")["Profit"].transform("sum")

print(len(df["Category_Avg_Sales"]) == len(df))
print(len(df["Region_Total_Profit"]) == len(df))

df[["Category", "Sales", "Category_Avg_Sales",
    "Region", "Profit", "Region_Total_Profit"]].head()

True
True


,Category,Sales,Category_Avg_Sales,Region,Profit,Region_Total_Profit
Order Date,,,,,,
2016-11-08,Furniture,261.9600,349.834887,South,41.9136,46749.4303
2016-11-08,Furniture,731.9400,349.834887,South,219.5820,46749.4303
2016-06-12,Office Supplies,14.6200,119.324101,West,6.8714,108418.4489
2015-10-11,Furniture,957.5775,349.834887,South,-383.0310,46749.4303
2015-10-11,Office Supplies,22.3680,119.324101,South,2.5164,46749.4303


### Task 4.2: Standardization within Groups (5 points)

1. Create 'Sales_Zscore_by_Category': standardize Sales within each Category
   Formula: (value - group_mean) / group_std
2. Create 'Profit_Pct_of_Region': each row's profit as percentage of its region's total
3. Rank products by Sales within each Sub-Category

In [20]:
df["Sales_Zscore_by_Category"] = df.groupby("Category")["Sales"].transform(
    lambda x: (x - x.mean()) / x.std()
)

df["Profit_Pct_of_Region"] = (
    df["Profit"] / df.groupby("Region")["Profit"].transform("sum")
) * 100

df["Sales_Rank_in_SubCategory"] = df.groupby("Sub-Category")["Sales"].rank(
    ascending=False,
    method="dense"
)

df[[
    "Category",
    "Sales",
    "Sales_Zscore_by_Category",
    "Region",
    "Profit",
    "Profit_Pct_of_Region",
    "Sub-Category",
    "Sales_Rank_in_SubCategory"
]].head()

,Category,Sales,Sales_Zscore_by_Category,Region,Profit,Profit_Pct_of_Region,Sub-Category,Sales_Rank_in_SubCategory
Order Date,,,,,,,,
2016-11-08,Furniture,261.9600,-0.174639,South,41.9136,0.089656,Bookcases,120.0
2016-11-08,Furniture,731.9400,0.759382,South,219.5820,0.469700,Chairs,115.0
2016-06-12,Office Supplies,14.6200,-0.273964,West,6.8714,0.006338,Labels,107.0
2015-10-11,Furniture,957.5775,1.207806,South,-383.0310,-0.819328,Tables,62.0
2015-10-11,Office Supplies,22.3680,-0.253691,South,2.5164,0.005383,Storage,559.0


### Task 4.3: Cumulative Operations (5 points)

1. Calculate cumulative Sales by Customer (running total per customer)
2. Calculate the rank of each order by Sales within each Customer
3. Calculate the cumulative count of orders per Region

In [21]:
df["Cumulative_Sales_by_Customer"] = df.groupby("Customer Name")["Sales"].cumsum()

df["Sales_Rank_by_Customer"] = df.groupby("Customer Name")["Sales"].rank(
    ascending=False,
    method="dense"
)

df["Cumulative_Orders_by_Region"] = df.groupby("Region").cumcount() + 1

df[[
    "Customer Name",
    "Sales",
    "Cumulative_Sales_by_Customer",
    "Sales_Rank_by_Customer",
    "Region",
    "Cumulative_Orders_by_Region"
]].head()

,Customer Name,Sales,Cumulative_Sales_by_Customer,Sales_Rank_by_Customer,Region,Cumulative_Orders_by_Region
Order Date,,,,,,
2016-11-08,Claire Gute,261.9600,261.9600,2.0,South,1
2016-11-08,Claire Gute,731.9400,993.9000,1.0,South,2
2016-06-12,Darrin Van Huff,14.6200,14.6200,7.0,West,1
2015-10-11,Sean O'Donnell,957.5775,957.5775,1.0,South,3
2015-10-11,Sean O'Donnell,22.3680,979.9455,11.0,South,4


---
## Part 5: Pivot Tables (15 points)
---

### Task 5.1: Basic Pivot Tables (5 points)

1. Create pivot table: Sales by Region (rows) and Category (columns)
2. Create pivot table: Average Profit by Segment (rows) and Year (columns)
3. Add margins (totals) to the first pivot table

In [22]:
pivot_sales = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="sum"
)

pivot_profit = pd.pivot_table(
    df,
    values="Profit",
    index="Segment",
    columns="Order_Year",
    aggfunc="mean"
)

pivot_sales_total = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="sum",
    margins=True
)

print(pivot_sales)

print(pivot_profit)

print(pivot_sales_total)

Category    Furniture  Office Supplies  Technology
Region                                            
Central   163797.1638       167026.415  170416.312
East      208291.2040       205516.055  264973.981
South     117298.6840       125651.313  148771.908
West      252612.7435       220853.249  251991.832
Order_Year        2014       2015       2016       2017
Segment                                                
Consumer     22.728832  25.297926  26.935959  27.319088
Corporate    22.116656  32.528813  39.085963  27.328942
Home Office  37.534765  36.569245  32.251185  31.760041
Category    Furniture  Office Supplies  Technology           All
Region                                                          
Central   163797.1638       167026.415  170416.312  5.012399e+05
East      208291.2040       205516.055  264973.981  6.787812e+05
South     117298.6840       125651.313  148771.908  3.917219e+05
West      252612.7435       220853.249  251991.832  7.254578e+05
All       741999.7953   

### Task 5.2: Multi-Aggregation Pivot Tables (5 points)

Create a pivot table showing:

- Rows: Category
- Columns: Region
- Values: Both sum and mean of Sales

In [23]:
pivot_multi = pd.pivot_table(
    df,
    values="Sales",
    index="Category",
    columns="Region",
    aggfunc=["sum", "mean"]
)

print(pivot_multi)

                         sum                                             mean  \
Region               Central        East       South         West     Central   
Category                                                                        
Furniture        163797.1638  208291.204  117298.684  252612.7435  340.534644   
Office Supplies  167026.4150  205516.055  125651.313  220853.2490  117.458801   
Technology       170416.3120  264973.981  148771.908  251991.8320  405.753124   

                                                     
Region                 East       South        West  
Category                                             
Furniture        346.574383  353.309289  357.302325  
Office Supplies  120.044425  126.282727  116.422377  
Technology       495.278469  507.753952  420.687533  


### Task 5.3: Crosstab Analysis (5 points)

1. Create a crosstab of Region vs Category (counts)
2. Create a crosstab of Segment vs Ship Mode with normalized values (by row)
3. What percentage of Corporate customers use Standard Class shipping?

In [25]:
df_temp = df.reset_index()

region_category = pd.crosstab(
    df_temp["Region"],
    df_temp["Category"]
)

segment_shipmode = pd.crosstab(
    df_temp["Segment"],
    df_temp["Ship Mode"],
    normalize="index"
)

corporate_standard_percentage = (
    segment_shipmode.loc["Corporate", "Standard Class"] * 100
)

print(region_category)

print(segment_shipmode)

print("Percentage of Corporate customers using Standard Class:")
print(corporate_standard_percentage)

Category  Furniture  Office Supplies  Technology
Region                                          
Central         481             1422         420
East            601             1712         535
South           332              995         293
West            707             1897         599
Ship Mode    First Class  Same Day  Second Class  Standard Class
Segment                                                         
Consumer        0.148141  0.061067      0.196494        0.594298
Corporate       0.160596  0.037748      0.201656        0.600000
Home Office     0.159282  0.062815      0.177229        0.600673
Percentage of Corporate customers using Standard Class:
60.0


---
## Part 6: Time Series Analysis (20 points)
---

### Task 6.1: Time-Based Selection (5 points)

Using the datetime index:

1. Select all orders from 2017
2. Select orders from Q4 of any year
3. Select orders between March and June 2016

In [27]:
orders_2017 = df.loc["2017"]

print(orders_2017.head())


orders_q4 = df[df.index.quarter == 4]

print(orders_q4.head())


orders_mar_jun_2016 = df[
    (df.index >= "2016-03-01") &
    (df.index <= "2016-06-30")
]

print(orders_mar_jun_2016.head())

            Row ID        Order ID  Ship Date       Ship Mode Customer ID  \
Order Date                                                                  
2017-04-15      13  CA-2017-114412 2017-04-20  Standard Class    AA-10480   
2017-07-16      24  US-2017-156909 2017-07-18    Second Class    SF-20065   
2017-10-19      35  CA-2017-107727 2017-10-23    Second Class    MA-17560   
2017-09-10      42  CA-2017-120999 2017-09-15  Standard Class    LC-16930   
2017-09-19      44  CA-2017-139619 2017-09-23  Standard Class    ES-14080   

              Customer Name      Segment        Country          City  \
Order Date                                                              
2017-04-15     Andrew Allen     Consumer  United States       Concord   
2017-07-16  Sandra Flanagan     Consumer  United States  Philadelphia   
2017-10-19     Matt Abelman  Home Office  United States       Houston   
2017-09-10   Linda Cazamias    Corporate  United States    Naperville   
2017-09-19       Erin 

### Task 6.2: Resampling (5 points)

1. Resample Sales to monthly totals
2. Resample Profit to quarterly averages
3. Find the month with highest total Sales

In [29]:
df = df.sort_index()

monthly_sales = df["Sales"].resample("ME").sum()

print(monthly_sales.head())

quarterly_profit = df["Profit"].resample("QE").mean()

print(quarterly_profit.head())

highest_sales_month = monthly_sales.idxmax()
highest_sales_value = monthly_sales.max()

print(highest_sales_month)
print(highest_sales_value)

Order Date
2014-01-31    14236.895
2014-02-28     4519.892
2014-03-31    55691.009
2014-04-30    28295.345
2014-05-31    23648.287
Freq: ME, Name: Sales, dtype: float64
Order Date
2014-03-31    13.514996
2014-06-30    28.581809
2014-09-30    22.703407
2014-12-31    28.773449
2015-03-31    35.634391
Freq: QE-DEC, Name: Profit, dtype: float64
2017-11-30 00:00:00
118447.825


### Task 6.3: Rolling Windows (5 points)

1. Calculate 7-day rolling average of Sales
2. Calculate 30-day rolling sum of Quantity
3. Calculate 90-day rolling standard deviation of Profit

In [30]:
sales_7day_avg = df["Sales"].rolling("7D").mean()

print(sales_7day_avg.head())

quantity_30day_sum = df["Quantity"].rolling("30D").sum()

print(quantity_30day_sum.head())

profit_90day_std = df["Profit"].rolling("90D").std()

print(profit_90day_std.head())

Order Date
2014-01-03     16.448000
2014-01-04     14.116000
2014-01-04    100.322667
2014-01-04     76.127000
2014-01-05     64.808800
Name: Sales, dtype: float64
Order Date
2014-01-03     2.0
2014-01-04     5.0
2014-01-04     8.0
2014-01-04    10.0
2014-01-05    13.0
Name: Quantity, dtype: float64
Order Date
2014-01-03          NaN
2014-01-04     0.904743
2014-01-04    40.238461
2014-01-04    33.475015
2014-01-05    30.337803
Name: Profit, dtype: float64


### Task 6.4: Shift and Lag Features (5 points)

1. Create a column showing previous month's total Sales (lag 1 month)
2. Create a column showing Sales change from previous month
3. Create a column showing Sales percentage change from previous month

In [31]:
monthly_sales = df["Sales"].resample("ME").sum()

previous_month_sales = monthly_sales.shift(1)

sales_change = monthly_sales - previous_month_sales

sales_percentage_change = monthly_sales.pct_change() * 100

print(previous_month_sales.head())

print(sales_change.head())

print(sales_percentage_change.head())

Order Date
2014-01-31          NaN
2014-02-28    14236.895
2014-03-31     4519.892
2014-04-30    55691.009
2014-05-31    28295.345
Freq: ME, Name: Sales, dtype: float64
Order Date
2014-01-31          NaN
2014-02-28    -9717.003
2014-03-31    51171.117
2014-04-30   -27395.664
2014-05-31    -4647.058
Freq: ME, Name: Sales, dtype: float64
Order Date
2014-01-31            NaN
2014-02-28     -68.252263
2014-03-31    1132.131409
2014-04-30     -49.192257
2014-05-31     -16.423401
Freq: ME, Name: Sales, dtype: float64


---
## Part 7: Business Analysis Questions (5 points)
---

Answer these business questions with code:

1. Which Region-Category combination has the highest profit margin?
2. Is there seasonality in sales? Which quarter performs best?
3. What is the year-over-year growth rate for each Category?
4. Which customers have increasing purchase trends?
5. What percentage of total profit comes from each Segment?

In [32]:
profit_margin = (
    df.groupby(["Region", "Category"])
    .apply(lambda x: (x["Profit"].sum() / x["Sales"].sum()) * 100)
)

print(profit_margin.sort_values(ascending=False).head(1))


quarter_sales = df.groupby("Order_Quarter")["Sales"].sum()

print(quarter_sales)
print(quarter_sales.idxmax())


year_category_sales = (
    df.groupby(["Order_Year", "Category"])["Sales"]
    .sum()
    .unstack()
)

yoy_growth = year_category_sales.pct_change() * 100

print(yoy_growth)


customer_year_sales = (
    df.groupby(["Customer Name", "Order_Year"])["Sales"]
    .sum()
    .unstack()
)

increasing_customers = customer_year_sales[
    customer_year_sales.diff(axis=1).iloc[:, 1:].gt(0).all(axis=1)
]

print(increasing_customers.index)


segment_profit_percentage = (
    df.groupby("Segment")["Profit"].sum()
    / df["Profit"].sum()
    * 100
)

print(segment_profit_percentage)

Region  Category       
West    Office Supplies    23.82118
dtype: float64
Order_Quarter
1    359681.5758
2    445509.6196
3    613932.1057
4    878077.5592
Name: Sales, dtype: float64
4
Category    Furniture  Office Supplies  Technology
Order_Year                                        
2014              NaN              NaN         NaN
2015         8.477093        -9.581824   -7.130049
2016        16.645257        34.034351   39.060729
2017         8.288444        33.792106   20.041435
Index(['Bruce Geld', 'Clytie Kelty', 'Cyma Kinney', 'Denise Monton',
       'Erica Smith', 'Fred Chung', 'Henry Goldwyn', 'John Castell',
       'Ken Brennan', 'Maria Bertelson', 'Mark Van Huff', 'Max Engle',
       'Nathan Cano', 'Noah Childs', 'Patrick O'Brill', 'Pauline Johnson',
       'Raymond Messe', 'Ross Baird'],
      dtype='str', name='Customer Name')
Segment
Consumer       46.829820
Corporate      32.115953
Home Office    21.054227
Name: Profit, dtype: float64


### Your Findings:

1. Best Region-Category:
2. Best Quarter:
3. YoY Growth:
4. Growing Customers:
5. Profit by Segment:

---
## Bonus: Executive Dashboard Data (10 points)
---

Create a comprehensive summary DataFrame that could power an executive dashboard:

1. Monthly KPIs: Sales, Profit, Orders, Avg Order Value
2. Include MoM (month-over-month) change percentages
3. Include YTD (year-to-date) cumulative totals
4. Include 3-month rolling averages

In [33]:
dashboard = pd.DataFrame()

dashboard["Sales"] = df["Sales"].resample("ME").sum()
dashboard["Profit"] = df["Profit"].resample("ME").sum()
dashboard["Orders"] = df["Order ID"].resample("ME").count()

dashboard["Avg_Order_Value"] = (
    dashboard["Sales"] / dashboard["Orders"]
)

dashboard["Sales_MoM_%"] = dashboard["Sales"].pct_change() * 100
dashboard["Profit_MoM_%"] = dashboard["Profit"].pct_change() * 100
dashboard["Orders_MoM_%"] = dashboard["Orders"].pct_change() * 100

dashboard["Sales_YTD"] = dashboard["Sales"].groupby(dashboard.index.year).cumsum()
dashboard["Profit_YTD"] = dashboard["Profit"].groupby(dashboard.index.year).cumsum()
dashboard["Orders_YTD"] = dashboard["Orders"].groupby(dashboard.index.year).cumsum()

dashboard["Sales_3M_Rolling_Avg"] = dashboard["Sales"].rolling(3).mean()
dashboard["Profit_3M_Rolling_Avg"] = dashboard["Profit"].rolling(3).mean()
dashboard["Orders_3M_Rolling_Avg"] = dashboard["Orders"].rolling(3).mean()

print(dashboard.head())
print(dashboard.shape)

                Sales     Profit  Orders  Avg_Order_Value  Sales_MoM_%  \
Order Date                                                               
2014-01-31  14236.895  2450.1907      79       180.213861          NaN   
2014-02-28   4519.892   862.3084      46        98.258522   -68.252263   
2014-03-31  55691.009   498.7299     157       354.719803  1132.131409   
2014-04-30  28295.345  3488.8352     135       209.595148   -49.192257   
2014-05-31  23648.287  2738.7096     122       193.838418   -16.423401   

            Profit_MoM_%  Orders_MoM_%   Sales_YTD  Profit_YTD  Orders_YTD  \
Order Date                                                                   
2014-01-31           NaN           NaN   14236.895   2450.1907          79   
2014-02-28    -64.806478    -41.772152   18756.787   3312.4991         125   
2014-03-31    -42.163395    241.304348   74447.796   3811.2290         282   
2014-04-30    599.544022    -14.012739  102743.141   7300.0642         417   
2014-05-31   

---
## Submission Checklist

- [ ] All groupby operations completed correctly
- [ ] Transform vs agg distinction demonstrated
- [ ] Pivot tables created with proper structure
- [ ] Time series operations (resample, rolling, shift) working
- [ ] Business questions answered with insights
- [ ] Code is clean and commented

**Total Points: 100 (+ 10 bonus)**